# 텍스트 임베딩 — `embed()`

기본 임베딩, 배치, task_type, 차원 축소, 코사인 유사도 계산.

In [3]:
from llm_request._internal import handler_client as llm_handler
from llm_request._internal.handler_client import embed

# llm_handler.configure(base_url="http://localhost:8100")


### 기본 임베딩

In [4]:
from llm_request._internal.handler_client import embed

result = embed("키움증권의 2024년 실적은?")

# 응답 구조 확인
print(f"모델: {result.model}")
print(f"request_id: {result.request_id}")
print(f"임베딩 수: {len(result.embeddings)}")
print(f"벡터 차원: {len(result.embeddings[0].values)}")
print(f"벡터 앞 5개: {result.embeddings[0].values[:5]}")

모델: gemini-embedding-001
request_id: d81fd45e-9110-42f4-a007-26340a4836ff
임베딩 수: 1
벡터 차원: 3072
벡터 앞 5개: [-0.01047961, 0.0061386074, 0.0002604844, -0.079388924, -0.024139348]


### 배치 임베딩

여러 텍스트를 한 번에 임베딩. 입력 순서 = 출력 순서.

In [5]:
result = embed(["키움증권 실적", "삼성전자 배당 정책", "코스피 전망"])

print(f"임베딩 수: {len(result.embeddings)}")
for i, emb in enumerate(result.embeddings):
    print(f"  [{i}] 차원: {len(emb.values)}, 앞 3개: {emb.values[:3]}")

임베딩 수: 3
  [0] 차원: 3072, 앞 3개: [-0.008738718, 0.018935014, 0.013339654]
  [1] 차원: 3072, 앞 3개: [1.3782818e-06, 0.049134586, 0.021774607]
  [2] 차원: 3072, 앞 3개: [-0.0022309595, -0.0017670715, 0.0004374371]


### 통계 확인

`statistics`는 Gemini API 응답에 따라 `None`일 수 있다.

In [6]:
result = embed("안녕하세요")

emb = result.embeddings[0]
print(f"벡터 차원: {len(emb.values)}")
if emb.statistics:
    print(f"토큰 수: {emb.statistics.token_count}")
    print(f"잘림 여부: {emb.statistics.truncated}")
else:
    print("statistics: None (Developer API에서는 미반환될 수 있음)")

벡터 차원: 3072
statistics: None (Developer API에서는 미반환될 수 있음)


### task_type — 용도별 최적화

검색 시스템에서 쿼리와 문서를 다르게 임베딩하면 품질이 향상된다.

In [ ]:
# 검색 쿼리 임베딩
query_result = embed("키움증권 실적", task_type="RETRIEVAL_QUERY")
print(f"query 벡터 차원: {len(query_result.embeddings[0].values)}")

# 문서 임베딩 (title 사용 가능)
doc_result = embed(
    "키움증권은 2024년 영업이익이 전년 대비 30% 증가했습니다.",
    task_type="RETRIEVAL_DOCUMENT",
    title="키움증권 2024년 실적 리포트",
)
print(f"document 벡터 차원: {len(doc_result.embeddings[0].values)}")

### 차원 축소

`output_dimensionality`로 벡터 크기를 줄여 저장 비용 절감.

In [ ]:
# 기본 차원 (3072)
full = embed("테스트")
print(f"기본 차원: {len(full.embeddings[0].values)}")

# 768 차원으로 축소
reduced = embed("테스트", output_dimensionality=768)
print(f"축소 차원: {len(reduced.embeddings[0].values)}")

### 코사인 유사도 계산

임베딩 벡터 간 유사도를 측정하는 예시.

In [ ]:
import numpy as np

result = embed(
    ["키움증권 실적 분석", "키움증권 2024년 영업이익", "오늘 날씨가 좋다"],
    task_type="SEMANTIC_SIMILARITY",
)

vecs = [np.array(e.values) for e in result.embeddings]

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"'실적 분석' vs '영업이익': {cosine_sim(vecs[0], vecs[1]):.4f}")  # 높은 유사도
print(f"'실적 분석' vs '날씨':     {cosine_sim(vecs[0], vecs[2]):.4f}")  # 낮은 유사도